# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source

The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```


In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the FAIR^2 dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata (do not subscript; access as an object)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Dataset Identifier: {metadata.identifier}")
print(f"Date Published: {metadata.datePublished}")

# Additional metadata
print("\nKeywords:", metadata.keywords)
print("\nLicense:", metadata.license)
print("\nSpatial Coverage:", metadata.spatialCoverage)

# The Croissant schema provides rich metadata about collection bias, limitations, scope, etc.
print("\nData Biases:", metadata.dataBiases)
print("\nData Collection:", metadata.dataCollection)
print("\nData Limitations:", metadata.dataLimitations)


## 2. Data Overview

Review available record sets, fields, and their IDs.

For exploring Croissant datasets, it's important to reference entities by their `@id`.

Let's inspect all record sets and their IDs, then review the fields within each record set.

In [ ]:
# List available record sets in the metadata
record_sets = []

if hasattr(metadata, 'recordSet') and metadata.recordSet:
    for rs in metadata.recordSet:
        if hasattr(rs, '@id'):
            record_sets.append(rs['@id'])
        elif hasattr(rs, '_id'):
            record_sets.append(rs._id)
        else:
            record_sets.append(rs)
else:
    # Try to infer record set IDs from the Croissant structure
    print("No record sets directly listed in metadata. Attempt to load from dataset object.")
    if hasattr(dataset, 'record_set_ids'):
        record_sets = dataset.record_set_ids
    else:
        try:
            # Fallback: extract from dataset internal representation
            record_sets = [r['@id'] for r in dataset._metadata['recordSet']]
        except Exception:
            print("Unable to extract record set IDs.")

# Print available record sets and their IDs
print("Record Sets (@id):")
for rsid in record_sets:
    print(rsid)

# For each record set, list available fields and their IDs
for record_set_id in record_sets:
    print(f"\nFields for Record Set {record_set_id}:")
    rs_metadata = dataset.record_set_metadata(record_set_id)
    if hasattr(rs_metadata, 'field') and rs_metadata.field:
        for field in rs_metadata.field:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else (field._id if hasattr(field, '_id') else str(field))
            field_name = field['name'] if isinstance(field, dict) and 'name' in field else (field.name if hasattr(field, 'name') else str(field_id))
            print(f"  {field_id}: {field_name}")
    else:
        print("  No fields found.")


In [ ]:
# For exploration, print some example records from the first record set
if record_sets:
    sample_record_set_id = record_sets[0]
    print(f"\nSample records from Record Set: {sample_record_set_id}")
    for i, x in enumerate(dataset.records(record_set=sample_record_set_id)):
        print(x)
        if i >= 2:
            break


## 3. Data Extraction

Load data from each record set into a DataFrame for analysis.

We reference each record set and field by its `@id`.

In [ ]:
# Extract data from each record set (@id)
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\nDataFrame for Record Set {record_set_id}:")
    print(df.head())

# Example: Show available columns for the first record set
if record_sets:
    example_rs_id = record_sets[0]
    print(f"\nColumns in DataFrame ({example_rs_id}):")
    print(dataframes[example_rs_id].columns.tolist())


## 4. Exploratory Data Analysis (EDA)

Apply common data processing: filter records, normalize numeric fields, and group data by attributes.

**Note:** All references to dataset entities use their `@id` according to Croissant schema best practices.

*Examples: Remove records with outlier values, normalize hormone levels, group by phenotype or clinical feature.*

In [ ]:
# Choose an example numeric field for EDA
# For illustration, select the first numeric column from the first record set

example_df = dataframes[example_rs_id]

# Find numeric columns
numeric_cols = example_df.select_dtypes(include='number').columns
numeric_field_id = numeric_cols[0] if len(numeric_cols) > 0 else None

if numeric_field_id:
    print(f"Selected numeric field for analysis: {numeric_field_id}")
    threshold = example_df[numeric_field_id].mean()  # Use mean as threshold example
    filtered_df = example_df[example_df[numeric_field_id] > threshold]

    print(f"\nFiltered records where {numeric_field_id} > mean ({threshold:.2f}):")
    print(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Attempt to group by a categorical field
    cat_cols = example_df.select_dtypes(include=['object']).columns
    group_field_id = cat_cols[0] if len(cat_cols) > 0 else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric columns found for analysis.")

## 5. Visualization

Visualize distributions or relationships between fields.

*Example: Plot histogram for hormone levels, or scatter vs clinical phenotype.*

In [ ]:
# Visualize numeric field distribution
if numeric_field_id:
    plt.figure(figsize=(6, 4))
    example_df[numeric_field_id].hist(bins=10, edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id} in Record Set {example_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Scatter plot example (if at least one categorical/group field and numeric)
    if group_field_id:
        plt.figure(figsize=(6, 4))
        for group, group_df in example_df.groupby(group_field_id):
            plt.scatter([group]*len(group_df[numeric_field_id]), group_df[numeric_field_id], label=str(group))
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.legend()
        plt.show()

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to:

- Load the FAIR^2 dataset using its Croissant schema URL.
- Explore dataset metadata, record sets, and fields via their `@id`.
- Extract tables into DataFrames and previewed their content.
- Performed basic EDA: filtering, normalization, and grouping.
- Visualized distributions of numeric fields and their relationship to clinical or genetic groups.

This approach highlights reproducible data exploration using Croissant schemas, ensuring FAIR interoperability and clarity by referencing all entities by `@id`.

*Due to the small sample size (N=3), most analyses are illustrative; results cannot be generalized but can be used for clinical or academic purposes.*